# 03 — Modeling: imbalance experiments, tuning, single-shot test

Implements **plan §8** (imbalance handling), §9 (model selection & tuning),
and §10 (evaluation).

Three blocks:

1. **Imbalance comparison** — fix XGBoost defaults and run baseline / class-weights /
   undersampling / SMOTE / hybrid on the validation set, ranked by PR-AUC.
2. **Optuna tuning** of the winning strategy via `src.train.run`.
3. **Single-shot test evaluation** with PR-AUC, recall@1%FPR, three thresholds,
   bootstrap CIs (§10.4), and PR + reliability plots.

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, roc_auc_score,
)
from sklearn.pipeline import Pipeline

from src.data import imbalance_ratio, load_raw, DEFAULT_DATA_PATH, split_stratified
from src.evaluate import bootstrap_metric, evaluate
from src.features import FeatureEngineer

sns.set_theme(context='notebook', style='whitegrid', palette='muted')
SEED = 42

## 1. Reload splits

In [ ]:
df = load_raw(ROOT / DEFAULT_DATA_PATH)
splits = split_stratified(df, random_state=SEED)
spw = imbalance_ratio(splits.y_train)
print(splits.summary().to_string(index=False))
print(f'scale_pos_weight = {spw:.2f}')

## 2. Imbalance experiments (plan §8)

Fix XGBoost defaults and the seed, vary only the imbalance strategy. Validate
on the natural-distribution val set (resampling applies to TRAIN only — §7.2).

In [ ]:
def fit_eval(label: str, X_tr, y_tr, X_val, y_val, scale_pos_weight=1.0):
    pipe = Pipeline(steps=[
        ('features', FeatureEngineer()),
        ('model', xgb.XGBClassifier(
            objective='binary:logistic', eval_metric='aucpr',
            tree_method='hist', n_estimators=400, learning_rate=0.1,
            max_depth=6, scale_pos_weight=scale_pos_weight,
            random_state=SEED, n_jobs=-1,
        )),
    ])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    fit_s = time.perf_counter() - t0
    proba = pipe.predict_proba(X_val)[:, 1]
    pr = average_precision_score(y_val, proba)
    roc = roc_auc_score(y_val, proba)
    return {'strategy': label, 'pr_auc_val': float(pr), 'roc_auc_val': float(roc),
            'fit_seconds': round(float(fit_s), 1)}

In [ ]:
fe_train = FeatureEngineer().fit(splits.X_train)
X_train_fe = fe_train.transform(splits.X_train)
y_train = splits.y_train.values

results = []
results.append(fit_eval('baseline', splits.X_train, splits.y_train,
                        splits.X_val, splits.y_val, scale_pos_weight=1.0))
results.append(fit_eval('class_weights', splits.X_train, splits.y_train,
                        splits.X_val, splits.y_val, scale_pos_weight=spw))

# Random undersampling: drop majority to 1:10.
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=SEED)
X_rus_fe, y_rus = rus.fit_resample(X_train_fe, y_train)
# Map back to raw frame for the Pipeline (it re-runs FE inside fit_eval).
# Simpler: feed the resampled raw rows by indexing splits.X_train with the
# sampler's selected indices via fit_resample on raw.
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=SEED)
X_rus, y_rus = rus.fit_resample(splits.X_train, splits.y_train)
results.append(fit_eval('undersample_1to10',
                        pd.DataFrame(X_rus, columns=splits.X_train.columns), pd.Series(y_rus),
                        splits.X_val, splits.y_val))

# SMOTE in the engineered feature space — synthesize then refit a fresh
# XGBClassifier directly on the engineered matrix (skipping FE since SMOTE
# already produced post-FE rows).
smote = SMOTE(sampling_strategy=0.1, random_state=SEED)
X_sm_fe, y_sm = smote.fit_resample(X_train_fe, y_train)
model_sm = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
    n_estimators=400, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1,
)
model_sm.fit(X_sm_fe, y_sm)
proba_sm = model_sm.predict_proba(fe_train.transform(splits.X_val))[:, 1]
results.append({'strategy': 'smote_1to10',
                'pr_auc_val': float(average_precision_score(splits.y_val, proba_sm)),
                'roc_auc_val': float(roc_auc_score(splits.y_val, proba_sm)),
                'fit_seconds': None})

# ADASYN.
ada = ADASYN(sampling_strategy=0.1, random_state=SEED)
X_ad_fe, y_ad = ada.fit_resample(X_train_fe, y_train)
model_ad = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
    n_estimators=400, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1,
)
model_ad.fit(X_ad_fe, y_ad)
proba_ad = model_ad.predict_proba(fe_train.transform(splits.X_val))[:, 1]
results.append({'strategy': 'adasyn_1to10',
                'pr_auc_val': float(average_precision_score(splits.y_val, proba_ad)),
                'roc_auc_val': float(roc_auc_score(splits.y_val, proba_ad)),
                'fit_seconds': None})

# SMOTE + Tomek hybrid.
smt = SMOTETomek(sampling_strategy=0.1, random_state=SEED)
X_st_fe, y_st = smt.fit_resample(X_train_fe, y_train)
model_st = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
    n_estimators=400, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1,
)
model_st.fit(X_st_fe, y_st)
proba_st = model_st.predict_proba(fe_train.transform(splits.X_val))[:, 1]
results.append({'strategy': 'smote_tomek_1to10',
                'pr_auc_val': float(average_precision_score(splits.y_val, proba_st)),
                'roc_auc_val': float(roc_auc_score(splits.y_val, proba_st)),
                'fit_seconds': None})

leaderboard = pd.DataFrame(results).sort_values('pr_auc_val', ascending=False).reset_index(drop=True)
leaderboard

Plan §8.3 expects class weights to match or beat SMOTE on this dataset. Our
leaderboard above is the empirical confirmation — keep both top strategies
into tuning per §8.2.

## 3. Tune the winning strategy via Optuna

Delegate to `src.train.run` to keep tuning logic in one place. Reduce trials
for notebook iteration; bump to 50–100 for the final run.

In [ ]:
from src.train import run as train_run
metadata = train_run(
    data_path=ROOT / DEFAULT_DATA_PATH,
    n_trials=20,           # bump to 50-100 for the portfolio run
    n_splits=5,
    seed=SEED,
    out_dir=ROOT / 'models',
    cost_fn=100.0,
    cost_fp=1.0,
)
print('Best CV PR-AUC:', metadata['model']['cv_pr_auc_mean'])
print('Test PR-AUC:   ', metadata['test_metrics']['pr_auc'])

## 4. Single-shot test evaluation

Reload the persisted champion and evaluate ONCE (plan §9.5 cardinal sin).

In [ ]:
import joblib
champion = joblib.load(ROOT / 'models' / 'champion.pkl')
proba_test = champion.predict_proba(splits.X_test)[:, 1]
report = evaluate(splits.y_test.to_numpy(), proba_test)
report.as_dict()

In [ ]:
# Bootstrap 95% CIs on PR-AUC and ROC-AUC (plan §10.4).
pr_pt, pr_lo, pr_hi = bootstrap_metric(splits.y_test.to_numpy(), proba_test, 'pr_auc', n_iter=1000, seed=SEED)
roc_pt, roc_lo, roc_hi = bootstrap_metric(splits.y_test.to_numpy(), proba_test, 'roc_auc', n_iter=1000, seed=SEED)
print(f'PR-AUC : {pr_pt:.4f}  95% CI [{pr_lo:.4f}, {pr_hi:.4f}]')
print(f'ROC-AUC: {roc_pt:.4f} 95% CI [{roc_lo:.4f}, {roc_hi:.4f}]')

In [ ]:
# Three thresholds reported side by side (plan §10.2).
rows = [t.as_dict() for t in report.thresholds.values()]
pd.DataFrame(rows)

## 5. PR curve & reliability diagram

In [ ]:
p, r, t = precision_recall_curve(splits.y_test, proba_test)
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(r, p)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall curve (PR-AUC={report.pr_auc:.4f})')
for name, tr in report.thresholds.items():
    ax.scatter([tr.recall], [tr.precision], s=70, label=f'{name} (t={tr.threshold:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Reliability diagram (plan §10.3): predicted probability vs observed frequency.
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(splits.y_test, proba_test, n_bins=10, strategy='quantile')
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], '--', color='grey', label='Perfectly calibrated')
ax.plot(mean_pred, frac_pos, marker='o', label='Champion')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability diagram (10 quantile bins)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Per-segment performance (plan §10.4)

Surfaces hidden weaknesses by Amount band and hour-of-day.

In [ ]:
test_df = splits.X_test.copy()
test_df['_y'] = splits.y_test.values
test_df['_p'] = proba_test
test_df['amount_band'] = pd.qcut(test_df['Amount'], q=4, duplicates='drop')
test_df['hour'] = ((test_df['Time'] // 3600) % 24).astype(int)

def seg_metrics(group: pd.DataFrame) -> pd.Series:
    if group['_y'].sum() == 0:
        return pd.Series({'n': len(group), 'frauds': 0, 'pr_auc': np.nan})
    return pd.Series({
        'n': len(group),
        'frauds': int(group['_y'].sum()),
        'pr_auc': float(average_precision_score(group['_y'], group['_p'])),
    })

by_band = test_df.groupby('amount_band', observed=True).apply(seg_metrics)
by_hour = test_df.groupby('hour').apply(seg_metrics)
print('By amount band:')
print(by_band)
print('\nBy hour-of-day:')
print(by_hour)